In [1]:
import json
import os
import random
from dotenv import load_dotenv
from google import genai
from google.genai import types
from rag_helper import RAGBase, documents
from tqdm.auto import tqdm

load_dotenv(override=True)
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
rag_system = RAGBase(index=None, llm_client=gemini_client)

# Sample 25 articles (with enough text to make good questions)
random.seed(42)
valid_docs = [
    d for d in documents if len(d.get("text") or d.get("content", "")) > 100
]
sample_docs = random.sample(valid_docs, min(25, len(valid_docs)))

print(f"Total articles: {len(documents)} | Sampled for evaluation: {len(sample_docs)}")
print("Sample item keys:", sample_docs[0].keys())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Total articles: 587 | Sampled for evaluation: 25
Sample item keys: dict_keys(['id', 'article_number', 'content', 'source', 'language'])


In [2]:
QUESTION_GEN_PROMPT = """
Vous simulez un salarié ou un employeur marocain qui pose des questions juridiques.
À partir de l'article suivant du Code du Travail marocain, générez 3 questions réalistes et concrètes auxquelles cet article répond.

RÈGLES IMPORTANTES :
1. NE MENTIONNEZ JAMAIS le numéro de l'article dans la question (le moteur de recherche doit le trouver tout seul).
2. Formulez des questions naturelles (ex: cas pratique, droits, obligations, délais).
3. Répondez UNIQUEMENT avec une liste JSON de 3 chaînes de caractères : ["question 1", "question 2", "question 3"]

ARTICLE {article_number} :
{text}
""".strip()

ground_truth = []

for doc in tqdm(sample_docs, desc="Generating Ground Truth Questions"):
    art_num = str(doc.get("article_number"))
    text = doc.get("text") or doc.get("content", "")

    prompt = QUESTION_GEN_PROMPT.format(article_number=art_num, text=text)

    try:
        response = gemini_client.models.generate_content(
            model=rag_system.model,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json", temperature=0.4
            ),
        )
        questions = json.loads(response.text)
        for q in questions:
            ground_truth.append({"question": q, "article_number": art_num})
    except Exception as e:
        print(f"Skipped Article {art_num}: {e}")

# Save to disk so we never have to regenerate it
with open("ground_truth.json", "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, ensure_ascii=False, indent=2)

print(f"✅ Generated {len(ground_truth)} test questions!")
print("Example:", ground_truth[0])

Generating Ground Truth Questions:   0%|          | 0/25 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Skipped Article 581: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 41.034453614s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit exceeded on Free Tier. See http

In [3]:
def search_bm25_only(query, num_results=5):
    res = rag_system.es_client.search(
        index=rag_system.index_name,
        query={
            "multi_match": {
                "query": query,
                "fields": ["text", "article_number^2"],
            }
        },
        size=num_results,
        _source=["article_number"],
    )
    return [hit["_source"] for hit in res["hits"]["hits"]]


def search_vector_only(query, num_results=5):
    q_vec = rag_system.embedding_model.encode(query).tolist()
    res = rag_system.es_client.search(
        index=rag_system.index_name,
        knn={
            "field": "text_vector",
            "query_vector": q_vec,
            "k": num_results,
            "num_candidates": 100,
        },
        _source=["article_number"],
    )
    return [hit["_source"] for hit in res["hits"]["hits"]]


def evaluate_search(search_fn, ground_truth_data):
    hits = 0
    mrr_sum = 0.0

    for item in tqdm(ground_truth_data, leave=False):
        expected_article = str(item["article_number"])
        results = search_fn(item["question"], num_results=5)
        retrieved_articles = [str(r["article_number"]) for r in results]

        if expected_article in retrieved_articles:
            hits += 1
            rank = retrieved_articles.index(expected_article) + 1
            mrr_sum += 1.0 / rank

    total = len(ground_truth_data)
    return {
        "Hit Rate @ 5": f"{(hits / total) * 100:.1f}%",
        "MRR @ 5": round(mrr_sum / total, 3),
    }


print("Evaluating 1/3: Keyword Search (BM25)...")
bm25_metrics = evaluate_search(search_bm25_only, ground_truth)

print("Evaluating 2/3: Vector Search (k-NN)...")
vector_metrics = evaluate_search(search_vector_only, ground_truth)

print("Evaluating 3/3: Hybrid Search (BM25 + Vector + RRF)...")
hybrid_metrics = evaluate_search(rag_system.search, ground_truth)

print("\n🏆 === BENCHMARK RESULTS === 🏆")
for name, metrics in [
    ("1. BM25 (Keyword Only)", bm25_metrics),
    ("2. Vector Search (k-NN Only)", vector_metrics),
    ("3. Hybrid Search (RRF)", hybrid_metrics),
]:
    print(
        f"{name:30} | Hit Rate: {metrics['Hit Rate @ 5']} | MRR: {metrics['MRR @ 5']}"
    )

Evaluating 1/3: Keyword Search (BM25)...


  0%|          | 0/72 [00:00<?, ?it/s]

Evaluating 2/3: Vector Search (k-NN)...


  0%|          | 0/72 [00:00<?, ?it/s]

Evaluating 3/3: Hybrid Search (BM25 + Vector + RRF)...


  0%|          | 0/72 [00:00<?, ?it/s]


🏆 === BENCHMARK RESULTS === 🏆
1. BM25 (Keyword Only)         | Hit Rate: 55.6% | MRR: 0.411
2. Vector Search (k-NN Only)   | Hit Rate: 66.7% | MRR: 0.5
3. Hybrid Search (RRF)         | Hit Rate: 75.0% | MRR: 0.535


In [4]:
# Cell 4: Optimize RRF Weights & k_rrf (Fast In-Memory Grid Search)
best_score = 0
best_params = {}

# Cache raw top-20 results once so grid search runs in 1 second
cached_runs = []
for item in tqdm(ground_truth, desc="Caching top-20 results"):
    q = item["question"]
    bm25_20 = search_bm25_only(q, num_results=20)
    vec_20 = search_vector_only(q, num_results=20)
    cached_runs.append(
        (str(item["article_number"]), bm25_20, vec_20)
    )

for k_rrf in [1, 5, 10, 20, 40, 60]:
    for vec_weight in [1.0, 1.2, 1.5, 2.0]:
        hits = 0
        mrr_sum = 0.0
        for expected_art, bm25_hits, vec_hits in cached_runs:
            scores = {}
            for rank, doc in enumerate(bm25_hits):
                art = str(doc["article_number"])
                scores[art] = scores.get(art, 0.0) + (
                    1.0 / (rank + 1 + k_rrf)
                )
            for rank, doc in enumerate(vec_hits):
                art = str(doc["article_number"])
                scores[art] = scores.get(art, 0.0) + (
                    vec_weight / (rank + 1 + k_rrf)
                )

            top_5 = sorted(
                scores.keys(), key=lambda x: scores[x], reverse=True
            )[:5]
            if expected_art in top_5:
                hits += 1
                mrr_sum += 1.0 / (top_5.index(expected_art) + 1)

        hit_rate = (hits / len(cached_runs)) * 100
        mrr = mrr_sum / len(cached_runs)

        if (hit_rate, mrr) > (
            best_score,
            best_params.get("mrr", 0),
        ):
            best_score = hit_rate
            best_params = {
                "k_rrf": k_rrf,
                "vec_weight": vec_weight,
                "hit_rate": f"{hit_rate:.1f}%",
                "mrr": round(mrr, 3),
            }

print("🔥 Optimized Hybrid RRF Parameters:", best_params)

Caching top-20 results:   0%|          | 0/72 [00:00<?, ?it/s]

🔥 Optimized Hybrid RRF Parameters: {'k_rrf': 1, 'vec_weight': 1.0, 'hit_rate': '77.8%', 'mrr': 0.573}


In [5]:
# Cell 5: LLM-as-a-Judge (End-to-End RAG Evaluation)
import time

JUDGE_PROMPT = """
Vous êtes un évaluateur expert en droit du travail marocain.
Évaluez la qualité de la RÉPONSE générée par le système RAG par rapport à la QUESTION et à l'ARTICLE DE RÉFÉRENCE.

QUESTION : {question}
ARTICLE DE RÉFÉRENCE ATTENDU : Article {article_number}
RÉPONSE DU SYSTÈME RAG : {answer}

Attribuez l'une des trois notes suivantes :
- "RELEVANT" : La réponse est exacte, complète et répond directement à la question juridique.
- "PARTLY_RELEVANT" : La réponse contient des éléments vrais mais est incomplète ou imprécise.
- "NON_RELEVANT" : Le système dit ne pas avoir l'information, se trompe, ou est hors-sujet.

Répondez UNIQUEMENT au format JSON :
{{"relevance": "RELEVANT | PARTLY_RELEVANT | NON_RELEVANT", "explanation": "Courte justification en 1 phrase"}}
""".strip()

eval_sample = ground_truth[:20]  # Test on 20 questions to respect API rate limits
judge_results = []

for item in tqdm(eval_sample, desc="Evaluating RAG Answers (LLM-as-a-Judge)"):
    q = item["question"]
    art = item["article_number"]

    # 1. Generate RAG answer (using optimized k_rrf=1)
    search_res = rag_system.search(q, num_results=5, k_rrf=1)
    prompt = rag_system.build_prompt(q, search_res)
    rag_answer = rag_system.llm(prompt)

    # 2. Judge the answer
    judge_input = JUDGE_PROMPT.format(
        question=q, article_number=art, answer=rag_answer
    )
    try:
        res = gemini_client.models.generate_content(
            model=rag_system.model,
            contents=judge_input,
            config=types.GenerateContentConfig(
                response_mime_type="application/json", temperature=0.0
            ),
        )
        verdict = json.loads(res.text)
        judge_results.append({
            "question": q,
            "expected_article": art,
            "answer": rag_answer,
            "relevance": verdict.get("relevance", "UNKNOWN"),
            "explanation": verdict.get("explanation", ""),
        })
    except Exception as e:
        print(f"Error judging question: {e}")

    time.sleep(1)  # Gentle pause for API rate limits

# Calculate summary percentages
counts = {}
for r in judge_results:
    rel = r["relevance"]
    counts[rel] = counts.get(rel, 0) + 1

total_judged = len(judge_results)
print("\n⚖️ === LLM-AS-A-JUDGE RESULTS === ⚖️")
for label in ["RELEVANT", "PARTLY_RELEVANT", "NON_RELEVANT"]:
    c = counts.get(label, 0)
    print(f"{label:18}: {c}/{total_judged} ({(c/total_judged)*100:.1f}%)")

Evaluating RAG Answers (LLM-as-a-Judge):   0%|          | 0/20 [00:00<?, ?it/s]

Error judging question: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 54.644764838s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit exceeded on Free Tier. See h

RetryError: RetryError[<Future at 0x79ca0ebe96d0 state=finished raised ClientError>]

In [6]:
# Resume from where it stopped without losing the first 55%!
print(f"Already judged: {len(judge_results)}/20 questions. Resuming...")

remaining_sample = eval_sample[len(judge_results) :]

for item in tqdm(remaining_sample, desc="Finishing remaining questions"):
    q = item["question"]
    art = item["article_number"]

    try:
        # 1. Generate RAG answer
        search_res = rag_system.search(q, num_results=5, k_rrf=1)
        prompt = rag_system.build_prompt(q, search_res)
        rag_answer = rag_system.llm(prompt)

        time.sleep(4)  # Wait 4s between Answer and Judge

        # 2. Judge the answer
        judge_input = JUDGE_PROMPT.format(
            question=q, article_number=art, answer=rag_answer
        )
        res = gemini_client.models.generate_content(
            model=rag_system.model,
            contents=judge_input,
            config=types.GenerateContentConfig(
                response_mime_type="application/json", temperature=0.0
            ),
        )
        verdict = json.loads(res.text)
        judge_results.append({
            "question": q,
            "expected_article": art,
            "answer": rag_answer,
            "relevance": verdict.get("relevance", "UNKNOWN"),
            "explanation": verdict.get("explanation", ""),
        })
    except Exception as e:
        print(f"Rate limit hit, waiting 30s... ({e})")
        time.sleep(30)

    time.sleep(5)  # Keep under 15 RPM free-tier limit

# Save results to disk so you never lose them
with open("judge_results.json", "w", encoding="utf-8") as f:
    json.dump(judge_results, f, ensure_ascii=False, indent=2)

# Print final summary
counts = {}
for r in judge_results:
    rel = r["relevance"]
    counts[rel] = counts.get(rel, 0) + 1

total_judged = len(judge_results)
print("\n⚖️ === LLM-AS-A-JUDGE RESULTS === ⚖️")
for label in ["RELEVANT", "PARTLY_RELEVANT", "NON_RELEVANT"]:
    c = counts.get(label, 0)
    print(f"{label:18}: {c}/{total_judged} ({(c/total_judged)*100:.1f}%)")

Already judged: 8/20 questions. Resuming...


Finishing remaining questions:   0%|          | 0/12 [00:00<?, ?it/s]


⚖️ === LLM-AS-A-JUDGE RESULTS === ⚖️
RELEVANT          : 16/20 (80.0%)
PARTLY_RELEVANT   : 1/20 (5.0%)
NON_RELEVANT      : 3/20 (15.0%)


In [7]:
for r in judge_results:
    if r["relevance"] == "NON_RELEVANT":
        print(f"Q: {r['question']} (Expected Art. {r['expected_article']})")
        print(f"A: {r['answer']}\n---")

Q: Est-ce qu'une entreprise peut compenser financièrement le non-respect du préavis par un salarié en retenant une partie de son indemnité de congé annuel ? (Expected Art. 255)
A: Oui, selon l'article 255 du Code du travail, lorsqu'un salarié rompt son contrat de travail sans respecter le préavis dû, l'employeur peut opérer une compensation entre l'indemnité du congé annuel payé et l'indemnité de préavis.
---
Q: Je travaille dans la même entreprise depuis 6 ans, ai-je droit à des jours de congé supplémentaires par rapport à la durée légale de base ? (Expected Art. 232)
A: Je n'ai pas l'information.
---
Q: Est-ce que mon ancienneté me permet d'augmenter mon nombre de jours de vacances annuelles, et si oui, y a-t-il une limite maximale ? (Expected Art. 232)
A: Je n'ai pas l'information.
---
